# 03 Machine Learning and Report Generation

**Project:** From Capital War to Fragmented Civilian Insecurity  
**Notebook role:** Build an explainable state-month risk model for civilian targeting and generate the final interactive HTML report.

The model is intentionally simple: a logistic classifier implemented in project code using NumPy. It predicts whether an admin1-month will have high civilian-targeting risk in the following month.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "code" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from project_utils import (
    PROCESSED_DIR,
    admin_to_geo_name,
    download_sudan_admin1_geojson,
    load_events,
    load_panel,
    save_table,
    train_evaluate_risk_model,
)
from Z_generate_report import (
    confusion_matrix_figure,
    ml_feature_figure,
    ml_risk_map,
    main as generate_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. Load the admin1-month panel

Each row is a Sudanese admin1-month. The target is based on next-month civilian-targeting events, which prevents the model from using future information in its predictors.

In [2]:
events = load_events()
panel = load_panel()

print(panel.shape)
display(panel.head())

target_summary = panel["target_high_civilian_targeting_next_month"].value_counts(dropna=False).to_frame("admin1_months")
display(target_summary)

(475, 47)


,admin1,macro_region,month,year_month,year,month_number,months_since_war_start,events,fatalities,civilian_targeting_events,violence_against_civilians_events,battles,explosions_remote,strategic_developments,protests_riots,saf_events,rsf_events,distinct_actor_count,unique_locations,unique_admin2,civilian_targeting_events_next_month,target_any_civilian_targeting_next_month,target_high_civilian_targeting_next_month,civilian_actor_events,fatal_event_count,mean_latitude,mean_longitude,events_lag1,fatalities_lag1,civilian_targeting_events_lag1,violence_against_civilians_events_lag1,battles_lag1,explosions_remote_lag1,strategic_developments_lag1,protests_riots_lag1,saf_events_lag1,rsf_events_lag1,civilian_actor_events_lag1,fatal_event_count_lag1,distinct_actor_count_lag1,unique_locations_lag1,unique_admin2_lag1,events_prev3mo,fatalities_prev3mo,civilian_targeting_events_prev3mo,battles_prev3mo,explosions_remote_prev3mo
0,Abyei,Abyei,2023-04-01,2023-04,2023,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6.0,1.0,0.0,0,0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Abyei,Abyei,2023-05-01,2023-05,2023,5,1,12,36,6,6,4,0,2,0,0,0,10,10,1,1.0,1.0,0.0,8,8,9.527508,28.551233,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Abyei,Abyei,2023-06-01,2023-06,2023,6,2,6,21,1,1,4,0,1,0,0,0,4,3,1,2.0,1.0,0.0,2,5,9.541467,28.515283,12,36,6,6,4,0,2,0,0,0,8,8,10,10,1,12,36,6,4,0
3,Abyei,Abyei,2023-07-01,2023-07,2023,7,3,2,0,2,2,0,0,0,0,0,0,4,1,1,1.0,1.0,0.0,2,0,9.589900,28.448600,6,21,1,1,4,0,1,0,0,0,2,5,4,3,1,18,57,7,8,0
4,Abyei,Abyei,2023-08-01,2023-08,2023,8,4,7,11,1,1,5,0,0,1,0,0,6,6,1,5.0,1.0,0.0,1,6,9.520714,28.536986,2,0,2,2,0,0,0,0,0,0,2,0,4,1,1,20,57,9,8,0


,admin1_months
target_high_civilian_targeting_next_month,
0.0,354
1.0,102
NaN,19


## 2. Train and evaluate the risk model

The model uses lagged and rolling conflict features: previous-month events, fatalities, civilian targeting, battles, remote violence, SAF/RSF involvement, actor diversity, location spread, and broad macro-region indicators.

In [3]:
ml = train_evaluate_risk_model(panel)
metrics = pd.DataFrame([ml["metrics"]])
display(metrics.T.rename(columns={0: "value"}))

display(ml["test_frame"][[
    "admin1", "year_month", "target", "predicted_probability", "predicted_high_risk"
]].sort_values("predicted_probability", ascending=False).head(20))

,value
threshold,0.64
accuracy,0.863158
precision,0.724138
recall,0.807692
f1,0.763636
auc,0.929766
true_positive,21
true_negative,61
false_positive,8
false_negative,5


,admin1,year_month,target,predicted_probability,predicted_high_risk
194,Khartoum,2024-11,1,1.000000,1
197,Khartoum,2025-02,1,1.000000,1
195,Khartoum,2024-12,1,1.000000,1
198,Khartoum,2025-03,1,1.000000,1
196,Khartoum,2025-01,1,1.000000,1
44,Al Jazirah,2024-11,1,1.000000,1
47,Al Jazirah,2025-02,1,1.000000,1
45,Al Jazirah,2024-12,1,0.999999,1
219,North Darfur,2024-11,1,0.999998,1
46,Al Jazirah,2025-01,1,0.999956,1


In [4]:
fig = confusion_matrix_figure(ml["metrics"])
fig.show()

## 3. Model interpretation

Because the model is logistic and features are standardized, larger absolute coefficients identify variables that are more influential in the predicted probability. Positive coefficients increase predicted risk; negative coefficients decrease it.

In [5]:
feature_importance = ml["feature_importance"]
display(feature_importance.head(20))

fig = ml_feature_figure(feature_importance)
fig.show()

,feature,coefficient,abs_coefficient
0,fatal_event_count_lag1,1.157768,1.157768
1,macro_Khartoum,1.134825,1.134825
2,macro_Eastern Sudan,-1.102024,1.102024
3,macro_Northern/Nile,-1.019375,1.019375
4,explosions_remote_lag1,0.926946,0.926946
5,fatalities_prev3mo,-0.882907,0.882907
6,civilian_actor_events_lag1,0.858851,0.858851
7,unique_admin2_lag1,0.717085,0.717085
8,battles_lag1,0.584959,0.584959
9,macro_Darfur,0.543191,0.543191


## 4. Latest available risk map

The current raw export ends in April 2025. The risk map therefore estimates next-month risk after the latest month in the available data, not a 2026 forecast.

In [6]:
geojson = download_sudan_admin1_geojson()
latest = ml["latest_predictions"].copy()
latest["geo_admin1"] = latest["admin1"].map(admin_to_geo_name)

display(latest[[
    "admin1", "year_month", "events", "fatalities", "civilian_targeting_events",
    "predicted_high_risk_probability", "predicted_high_risk"
]].head(12))

fig = ml_risk_map(latest, geojson)
fig.show()

,admin1,year_month,events,fatalities,civilian_targeting_events,predicted_high_risk_probability,predicted_high_risk
199,Khartoum,2025-04,73,262,27,0.999997,1
224,North Darfur,2025-04,81,1092,29,0.998671,1
249,North Kordofan,2025-04,41,199,10,0.962319,1
374,South Darfur,2025-04,20,27,4,0.909158,1
49,Al Jazirah,2025-04,3,4,1,0.707026,1
424,West Darfur,2025-04,5,0,2,0.521574,0
474,White Nile,2025-04,13,54,5,0.412452,0
74,Blue Nile,2025-04,3,13,1,0.325516,0
99,Central Darfur,2025-04,6,3,5,0.272753,0
124,East Darfur,2025-04,6,1,2,0.259546,0


## 5. Save model outputs

These CSVs are used by the final report and make the modeling step auditable.

In [7]:
save_table(ml["latest_predictions"], PROCESSED_DIR / "ml_latest_risk_admin1.csv")
save_table(ml["feature_importance"], PROCESSED_DIR / "ml_feature_importance.csv")
save_table(pd.DataFrame([ml["metrics"]]), PROCESSED_DIR / "ml_evaluation_metrics.csv")
save_table(ml["test_frame"], PROCESSED_DIR / "ml_test_predictions.csv")

print("Saved ML outputs to", PROCESSED_DIR)

Saved ML outputs to C:\Users\USER\Desktop\asm1\data\processed


## 6. Generate the interactive HTML report

The report combines narrative text, interactive maps, time-series charts, actor analysis, model interpretation, and references. It is saved in `docs/index.html` so it can be served with GitHub Pages later.

In [8]:
generate_report()
print("Report generated at", PROJECT_ROOT / "docs" / "index.html")

{
  "events": 16004,
  "date_min": "2023-04-15",
  "date_max": "2025-04-22",
  "html_report": "C:\\Users\\USER\\Desktop\\asm1\\docs\\index.html",
  "model_metrics": {
    "threshold": 0.6400000000000001,
    "accuracy": 0.8631578947368421,
    "precision": 0.7241379310344828,
    "recall": 0.8076923076923077,
    "f1": 0.7636363636363636,
    "auc": 0.9297658862876255,
    "true_positive": 21,
    "true_negative": 61,
    "false_positive": 8,
    "false_negative": 5,
    "train_rows": 361,
    "test_rows": 95,
    "test_start": "2024-11-01",
    "test_end": "2025-03-01",
    "positive_rate_train": 0.21052631578947367,
    "positive_rate_test": 0.2736842105263158
  }
}
Report generated at c:\Users\USER\Desktop\asm1\docs\index.html
